# CLIP ViT-L/14: Standard retrieval vs projection-rearrangement (SPEED only)
Standard: `img -> vision_model -> visual_projection -> cos(text_emb)`.
Proposed: `img -> vision_model -> raw(1024)`; query: `text_projection -> t @ W_v` (transpose of image projection, **not** inverse) and score against raw embeddings.

**Math note.** `cos(t, W v) = (W^T t)·v / (|t| · |W v|)`. The numerator is cheap. The denominator `|W v|` is a per-image norm of the *projected* embedding. Computing it exactly needs per-image work (done here through the Gram matrix `G = W^T W`, `|Wv|^2 = v^T G v`, a 1024x1024 product per image, which is **more** FLOPs than the 1024x768 projection). So two Proposed variants are timed:
- **Proposed (exact norm)**: reproduces CLIP cosine exactly; image-side norm time is reported as its own row.
- **Proposed (approx, raw-norm)**: divides by `|v|` instead of `|Wv|`. Lower bound on cost, NOT equal to CLIP scoring.

Turn on **Internet** (model download) and **GPU** in the Kaggle sidebar.

In [ ]:
import os, time, math, json
from contextlib import contextmanager
import numpy as np, pandas as pd, torch, cv2, transformers
from transformers import CLIPModel, CLIPProcessor

VIDEO_PATH = "/kaggle/input/datasets/seewhee1130/video-dataset-krishna/videoplayback.mp4"
MODEL_NAME = "openai/clip-vit-large-patch14"
QUERY = "lion"
TARGET_FPS = 10
EXPECTED_DURATION = 570.0     # spec value; measured value is what is actually used
DURATION_TOL = 5.0            # seconds
EXPECTED_FRAMES = 5700
FORCE_5700 = False            # True: uniformly sample exactly 5700 frames (effective fps != 10 if video is shorter)
BATCH_SIZE = 64
N_REPS = 5
WARMUP_BATCHES = 3
SMOKE_N = 32
DEVICE, DTYPE = "cuda", torch.float16
assert torch.cuda.is_available(), "Enable a GPU accelerator"

In [ ]:
# ---------- 2. Video validation (no model inference yet) ----------
cap = cv2.VideoCapture(VIDEO_PATH); assert cap.isOpened(), "cannot open video"
src_fps = cap.get(cv2.CAP_PROP_FPS)
src_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
src_w, src_h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
duration = src_frames / src_fps
print(f"Measured duration : {duration:.3f} s ({int(duration//60)}m {duration%60:.1f}s)")
print(f"Source FPS        : {src_fps:.4f}")
print(f"Source frame count: {src_frames}")
print(f"Source resolution : {src_w}x{src_h}")
print(f"Target sampling   : {TARGET_FPS} fps")
if abs(duration - EXPECTED_DURATION) > DURATION_TOL:
    print(f"WARNING: measured {duration:.1f}s differs from expected ~{EXPECTED_DURATION:.0f}s by {duration-EXPECTED_DURATION:+.1f}s. Proceeding with the MEASURED duration.")
if FORCE_5700:
    N_TARGET = EXPECTED_FRAMES
    indices = np.linspace(0, src_frames - 1, N_TARGET).round().astype(int).tolist()
    print(f"FORCE_5700: uniform spacing, effective sampling = {N_TARGET/duration:.4f} fps")
else:
    N_TARGET = int(round(duration * TARGET_FPS))
    indices = [min(int(round(i * src_fps / TARGET_FPS)), src_frames - 1) for i in range(N_TARGET)]
if N_TARGET != EXPECTED_FRAMES:
    print(f"NOTE: {duration:.1f}s x {TARGET_FPS} fps = {N_TARGET} frames (spec assumed {EXPECTED_FRAMES}). Using {N_TARGET}.")
print("Frames to extract :", N_TARGET)

In [ ]:
# ---------- Frame extraction + CLIP preprocessing (reported separately, not in model benchmark) ----------
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
def extract_and_preprocess(indices, chunk=256):
    cap = cv2.VideoCapture(VIDEO_PATH)
    out, buf, j, pos, n = [], [], 0, 0, len(indices)
    first_frame = None
    def flush():
        nonlocal buf
        if buf:
            pv = processor.image_processor(images=buf, return_tensors="pt")["pixel_values"]
            out.append(pv.to(DTYPE)); buf = []
    while j < n:
        if not cap.grab(): break
        fr = None
        while j < n and indices[j] == pos:
            if fr is None:
                ok, bgr = cap.retrieve(); assert ok
                fr = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
                if first_frame is None: first_frame = fr.shape
            buf.append(fr); j += 1
        pos += 1
        if len(buf) >= chunk: flush()
    cap.release(); flush()
    assert j == n, f"decoded only {j}/{n} frames"
    return torch.cat(out), first_frame

t0 = time.time()
pixels_cpu, frame_shape = extract_and_preprocess(indices)
prep_time = time.time() - t0
assert pixels_cpu.shape[0] == N_TARGET
pixels = pixels_cpu.to(DEVICE)       # all frames resident on GPU; identical tensor used by both methods
del pixels_cpu
print(f"Extracted frames: {pixels.shape[0]} | raw frame (HxWxC): {frame_shape} | model input: {tuple(pixels.shape[1:])} {pixels.dtype}")
print(f"Decode+preprocess time (separate preprocessing metric, NOT in benchmark): {prep_time:.1f}s")

In [ ]:
# ---------- 1. Model: four components accessed explicitly ----------
model = CLIPModel.from_pretrained(MODEL_NAME, torch_dtype=DTYPE).to(DEVICE).eval()
model.requires_grad_(False)
vision_model, visual_projection = model.vision_model, model.visual_projection
text_model, text_projection = model.text_model, model.text_projection
torch.cuda.synchronize()
model_mem_mb = torch.cuda.memory_allocated() / 2**20  # includes frames resident on GPU
frames_mem_mb = pixels.numel() * pixels.element_size() / 2**20
model_only_mb = model_mem_mb - frames_mem_mb

Wv = visual_projection.weight   # nn.Linear weight: (out=768, in=1024), y = x @ Wv.T, no bias
print("visual_projection.weight:", tuple(Wv.shape), "| bias:", visual_projection.bias)
print("text_projection.weight  :", tuple(text_projection.weight.shape), "| bias:", text_projection.bias)
assert Wv.shape == (768, 1024) and text_projection.weight.shape == (768, 768)
assert visual_projection.bias is None and text_projection.bias is None

# One-time model-side precompute for exact normalization (NOT timed per image; reported)
G = (Wv.float().T @ Wv.float()).to(DTYPE)   # (1024,1024), |Wv|^2 = v^T G v
print("Gram matrix G:", tuple(G.shape))

tok = CLIPProcessor.from_pretrained(MODEL_NAME).tokenizer
text_inputs = {k: v.to(DEVICE) for k, v in tok([QUERY], padding=True, return_tensors="pt").items()}  # tokenization outside timing

# ---------- 12. Reproducibility header ----------
p = torch.cuda.get_device_properties(0)
print("="*70)
print("Model              :", MODEL_NAME)
print("Transformers       :", transformers.__version__)
print("PyTorch            :", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU                :", p.name, f"| {p.total_memory/2**30:.1f} GiB")
print("Batch size         :", BATCH_SIZE, "| precision: FP16 | reps:", N_REPS, "| warmup batches:", WARMUP_BATCHES)
print(f"Video              : {duration:.2f}s, source {src_fps:.3f} fps, {src_frames} frames, {src_w}x{src_h}")
print("Extracted frames   :", pixels.shape[0], "| model input res:", tuple(pixels.shape[2:]))
print("Query              :", repr(QUERY))
print(f"GPU mem: model only {model_only_mb:.0f} MiB | frames resident {frames_mem_mb:.0f} MiB")
print("="*70)

In [ ]:
# ---------- Timing infrastructure: CUDA events, separate span per operation ----------
class Rec:
    def __init__(self): self.p = {}
    @contextmanager
    def t(self, name):
        a, b = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        a.record(); yield; b.record()
        self.p.setdefault(name, []).append((a, b))
    def totals(self):
        torch.cuda.synchronize()
        return {k: sum(a.elapsed_time(b) for a, b in v) for k, v in self.p.items()}   # ms

@torch.inference_mode()
def stage1_A(pix, bs, rec):
    outs = []
    with rec.t("s1_wall"):
        for i in range(0, len(pix), bs):
            x = pix[i:i+bs]
            with rec.t("vision"):   raw = vision_model(pixel_values=x).pooler_output       # (b,1024)
            with rec.t("img_proj"): e = visual_projection(raw)                             # (b,768)
            with rec.t("img_norm"): outs.append(e / e.norm(dim=-1, keepdim=True))          # image-side norm paid in Stage 1 (same as method B)
    return torch.cat(outs)   # already L2-normalized

@torch.inference_mode()
def stage1_B(pix, bs, rec):
    raws, norms = [], []
    with rec.t("s1_wall"):
        for i in range(0, len(pix), bs):
            x = pix[i:i+bs]
            with rec.t("vision"):   raw = vision_model(pixel_values=x).pooler_output       # (b,1024), NO projection
            with rec.t("img_norm"):                                                         # exact-norm variant only
                n2 = ((raw @ G).float() * raw.float()).sum(1)
                norms.append(n2.clamp_min(0).sqrt())
            raws.append(raw)
    return torch.cat(raws), torch.cat(norms)

@torch.inference_mode()
def stage2_A(img_emb, rec):
    with rec.t("text_enc"):  h = text_model(**text_inputs).pooler_output
    with rec.t("text_proj"): t = text_projection(h)                                         # (1,768)
    with rec.t("sim"):
        tn = t / t.norm(dim=-1, keepdim=True)
        s = (img_emb @ tn.T).squeeze(1)      # img_emb pre-normalized in Stage 1
    return s

@torch.inference_mode()
def stage2_B(raw, vnorm, rec):
    with rec.t("text_enc"):  h = text_model(**text_inputs).pooler_output
    with rec.t("text_proj"): t = text_projection(h)                                         # (1,768)
    with rec.t("query_xform"):
        q = t @ Wv                                                                          # (1,1024) = (W^T t)^T, transpose not inverse
        tnorm = t.float().norm()
    with rec.t("sim_exact"):
        s_exact = (raw @ q.squeeze(0)).float() / (vnorm * tnorm)
    with rec.t("sim_approx"):
        s_approx = (raw @ q.squeeze(0)).float() / (raw.float().norm(dim=-1) * tnorm)
    return s_exact, s_approx

def run_A(pix, bs):
    rec = Rec(); E = stage1_A(pix, bs, rec); s = stage2_A(E, rec); return rec, E, s
def run_B(pix, bs):
    rec = Rec(); R, N = stage1_B(pix, bs, rec); se, sa = stage2_B(R, N, rec); return rec, R, N, se, sa

In [ ]:
# ---------- 11. Smoke test ----------
print("Module paths used:")
print("  A image :", type(vision_model).__name__, "-> .pooler_output (1024) ->", type(visual_projection).__name__, tuple(Wv.shape))
print("  A text  :", type(text_model).__name__, "-> .pooler_output ->", type(text_projection).__name__)
print("  B image :", type(vision_model).__name__, "-> .pooler_output (1024), no projection")
print("  B query :", "text_projection(text_model(.)) @ visual_projection.weight  [(1,768)@(768,1024)]")

counts = {"visual_projection": 0, "text_projection": 0}
hooks = [visual_projection.register_forward_hook(lambda m,i,o: counts.__setitem__("visual_projection", counts["visual_projection"]+1)),
         text_projection.register_forward_hook(lambda m,i,o: counts.__setitem__("text_projection", counts["text_projection"]+1))]
sbs = min(16, BATCH_SIZE); sp = pixels[:SMOKE_N]
print("Same frames for both methods; checksum:", float(sp.float().sum()))

counts.update(visual_projection=0, text_projection=0)
recA, EA, sA = run_A(sp, sbs); totA = recA.totals()
assert counts["visual_projection"] == math.ceil(SMOKE_N/sbs), counts
print("A: visual_projection calls =", counts["visual_projection"], "(expected", math.ceil(SMOKE_N/sbs), ") | emb", tuple(EA.shape))

counts.update(visual_projection=0, text_projection=0)
recB, RB, NB, sBe, sBa = run_B(sp, sbs); totB = recB.totals()
assert counts["visual_projection"] == 0, "hidden image projection in method B!"
print("B: visual_projection calls =", counts["visual_projection"], "| raw emb", tuple(RB.shape), "| text_projection calls:", counts["text_projection"])
for h in hooks: h.remove()   # hooks removed so they never sit inside timed regions

assert all(x.is_cuda for x in (EA, RB, sA, sBe)), "CPU tensor found"
assert EA.shape[1] == 768 and RB.shape[1] == 1024
d_exact = (sA.float() - sBe).abs().max().item(); d_approx = (sA.float() - sBa).abs().max().item()
print(f"Sanity (math check only, not an accuracy study): max|cos_A - cos_B_exact| = {d_exact:.5f} | approx variant = {d_approx:.5f}")
assert d_exact < 2e-2, "exact-norm variant does not reproduce CLIP cosine; fix before full run"
print("Smoke timings (ms):", {k: round(v, 3) for k, v in totA.items()}, "\n", {k: round(v, 3) for k, v in totB.items()})
print("SMOKE TEST PASSED")
del EA, RB, NB

In [ ]:
# ---------- Full benchmark (reps interleaved A/B, order alternated to cancel clock/thermal drift) ----------
def warmup(fn):
    n = min(len(pixels), WARMUP_BATCHES * BATCH_SIZE)
    for _ in range(2): fn(pixels[:n], BATCH_SIZE)
    torch.cuda.synchronize()

warmup(run_A); warmup(run_B); torch.cuda.empty_cache()
resA, resB, peaks = [], [], {"A": [0.0, 0.0], "B": [0.0, 0.0]}
for r in range(N_REPS):
    order = [("A", run_A, resA), ("B", run_B, resB)]
    if r % 2: order.reverse()
    for k, fn, res in order:
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
        out = fn(pixels, BATCH_SIZE); res.append(out[0].totals()); del out
        peaks[k][0] = max(peaks[k][0], torch.cuda.max_memory_allocated()/2**20)
        peaks[k][1] = max(peaks[k][1], torch.cuda.max_memory_reserved()/2**20)
    print(f"rep {r+1}/{N_REPS} done")
memA = dict(peak_alloc_mb=peaks["A"][0], peak_reserved_mb=peaks["A"][1])
memB = dict(peak_alloc_mb=peaks["B"][0], peak_reserved_mb=peaks["B"][1])
dimA, dimB = 768, 1024
print(f"\nStage-2 dimensionality: standard = {dimA}, proposed = {dimB}")
json.dump(dict(A=resA, B=resB, memA=memA, memB=memB, duration=duration, n_frames=int(pixels.shape[0]),
               batch=BATCH_SIZE), open("/kaggle/working/results.json", "w"), indent=1)

In [ ]:
# ---------- 10. Tables ----------
def comps(res, kind, variant="exact"):
    rows = []
    for r in res:
        d = dict(vision=r["vision"], img_proj=r.get("img_proj", 0.0),
                 img_norm=r.get("img_norm", 0.0) if (kind == "A" or variant == "exact") else 0.0,
                 text_enc=r["text_enc"], text_proj=r["text_proj"], query=r.get("query_xform", 0.0),
                 sim=r["sim"] if kind == "A" else r["sim_" + variant])
        d["s1"] = d["vision"] + d["img_proj"] + d["img_norm"]
        d["s2"] = d["text_enc"] + d["text_proj"] + d["query"] + d["sim"]
        d["total"] = d["s1"] + d["s2"]
        rows.append(d)
    return pd.DataFrame(rows)

LABELS = [("vision","Vision encoder time"),("img_proj","Image projection time"),("img_norm","Image-side norm (baseline: |e|; proposed-exact: Gram |Wv|)"),
          ("s1","**Stage 1 total**"),("text_enc","Text encoder time"),("text_proj","Text projection time"),
          ("query","Query-side projection (+|t|)"),("sim","Similarity/retrieval time"),("s2","**Stage 2 total**"),("total","**Total pipeline time**")]

def table(variant):
    A, B = comps(resA, "A"), comps(resB, "B", variant)
    rows = []
    for k, lab in LABELS:
        a, b = A[k].median(), B[k].median()
        rows.append({"Metric": lab, "Standard CLIP (ms)": f"{a:.2f} (sd {A[k].std(ddof=1):.2f})",
                     "Proposed (ms)": f"{b:.2f} (sd {B[k].std(ddof=1):.2f})", "Difference (ms)": f"{b-a:+.2f}",
                     "Speedup": f"{a/b:.3f}x" if b > 0 and a > 0 else "n/a",
                     "Reduction %": f"{(a-b)/a*100:.2f}%" if a > 0 else "n/a"})
    ma, mb = memA["peak_alloc_mb"], memB["peak_alloc_mb"]
    rows.append({"Metric": "Peak GPU memory allocated (MiB)", "Standard CLIP (ms)": f"{ma:.0f}", "Proposed (ms)": f"{mb:.0f}",
                 "Difference (ms)": f"{mb-ma:+.0f}", "Speedup": "-", "Reduction %": f"{(ma-mb)/ma*100:.2f}%"})
    rows.append({"Metric": "Peak GPU memory reserved (MiB)", "Standard CLIP (ms)": f"{memA['peak_reserved_mb']:.0f}", "Proposed (ms)": f"{memB['peak_reserved_mb']:.0f}",
                 "Difference (ms)": f"{memB['peak_reserved_mb']-memA['peak_reserved_mb']:+.0f}", "Speedup": "-", "Reduction %": "-"})
    df = pd.DataFrame(rows).rename(columns={"Standard CLIP (ms)": "Standard CLIP", "Proposed (ms)": "Proposed Method", "Difference (ms)": "Difference"})
    return df, A, B

from IPython.display import display, Markdown
print(f"MEDIAN over reps | Frames: {pixels.shape[0]} | batch {BATCH_SIZE} | reps {N_REPS} | FP16 | model-only GPU memory {model_only_mb:.0f} MiB (+{frames_mem_mb:.0f} MiB resident frames)")
for variant, title in [("exact", "PROPOSED (EXACT CLIP NORMALIZATION)"), ("approx", "PROPOSED (APPROX: raw-norm, lower-bound cost, not CLIP-equivalent)")]:
    df, A, B = table(variant)
    display(Markdown(f"### {title}")); display(df)
    for k, lab in [("s1","Stage 1"),("s2","Stage 2"),("total","End-to-end")]:
        a, b = A[k].median(), B[k].median()
        print(f"{lab} speedup = {a:.2f} / {b:.2f} = {a/b:.4f}x | time reduction {(a-b)/a*100:.3f}% | saved {a-b:+.2f} ms")
    print(f"Per-image image-projection cost removed: {A['img_proj'].median()/pixels.shape[0]*1000:.2f} us | vision encoder share of baseline total: {A['vision'].median()/A['total'].median()*100:.1f}%\n")
print("Note: vision encoder runs identically in both methods; any difference in that row is run-to-run noise.")

In [ ]:
# ---------- Micro-benchmark: only ops that differ, on cached embeddings, median of many reps ----------
@torch.inference_mode()
def timeit(fn, reps=50, warm=10):
    for _ in range(warm): fn()
    torch.cuda.synchronize(); ts = []
    for _ in range(reps):
        a, b = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        a.record(); fn(); b.record(); torch.cuda.synchronize(); ts.append(a.elapsed_time(b))
    ts = np.array(ts); return np.median(ts), np.percentile(ts, 25), np.percentile(ts, 75)

with torch.inference_mode():
    _, R, vn, _, _ = run_B(pixels, BATCH_SIZE)
    batches = list(R.split(BATCH_SIZE))
    E = torch.cat([visual_projection(b) for b in batches]); E = E / E.norm(dim=-1, keepdim=True)
    t = text_projection(text_model(**text_inputs).pooler_output)
    tn_ = t / t.norm(dim=-1, keepdim=True); tnorm_ = t.float().norm()

def s1_A():  return [(lambda e: e / e.norm(dim=-1, keepdim=True))(visual_projection(b)) for b in batches]
def s1_Be(): return [((b @ G).float() * b.float()).sum(1).clamp_min(0).sqrt() for b in batches]
def s2_A():  return (E @ tn_.T).squeeze(1)
def s2_Be():
    q = t @ Wv; return (R @ q.squeeze(0)).float() / (vn * tnorm_)
def s2_Ba():
    q = t @ Wv; return (R @ q.squeeze(0)).float() / (R.float().norm(dim=-1) * tnorm_)

print(f"{'op':40s} median   [p25, p75]  (ms, full {len(R)}-frame pass)")
for name, f in [("S1 baseline: projection + normalize", s1_A), ("S1 proposed-exact: Gram norm", s1_Be),
                ("S2 baseline: E @ t_norm (768-d)", s2_A), ("S2 proposed-exact: q=t@Wv + score", s2_Be),
                ("S2 proposed-approx: q=t@Wv + score", s2_Ba)]:
    m, lo, hi = timeit(f); print(f"{name:40s} {m:7.3f}  [{lo:.3f}, {hi:.3f}]")